In [1]:
# =========================
# Imports
# =========================

import requests
import pandas as pd
from pathlib import Path
import datetime as dt

In [3]:
# =========================
# Container Bronze
# =========================

try:

    # ------------------------------------------
    # Widget Databricks
    # ------------------------------------------

    dbutils.widgets.text(
        "BRONZE_CONTAINER",
        "bronze"
    )

    BRONZE_CONTAINER = (
        dbutils.widgets.get(
            "BRONZE_CONTAINER"
        ) or "bronze"
    )

    print(
        "✅ Widget Databricks carregado."
    )

except NameError:

    # ------------------------------------------
    # Execução VSCode / Jupyter
    # ------------------------------------------

    print(
        "⚠️ dbutils não encontrado. "
        "Utilizando configuração local."
    )

    BRONZE_CONTAINER = "bronze"

# =========================
# Validação
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

⚠️ dbutils não encontrado. Utilizando configuração local.
✅ Container Bronze: bronze


In [4]:
# =========================
# Azure Blob
# =========================

from azure.storage.blob import BlobServiceClient

try:

    try:

        AZURE_STORAGE_ACCOUNT = (
            dbutils.secrets.get(
                scope="kvfiaptechprod",
                key="AZURE-STORAGE-ACCOUNT"
            )
        )

        AZURE_STORAGE_KEY = (
            dbutils.secrets.get(
                scope="kvfiaptechprod",
                key="AZURE-STORAGE-KEY"
            )
        )

    except NameError:

        import os

        AZURE_STORAGE_ACCOUNT = os.getenv(
            "AZURE_STORAGE_ACCOUNT"
        )

        AZURE_STORAGE_KEY = os.getenv(
            "AZURE_STORAGE_KEY"
        )

    blob_service_client = BlobServiceClient(
        account_url=(
            f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
        ),
        credential=AZURE_STORAGE_KEY
    )

    print(
        "✅ Cliente Azure Blob inicializado."
    )

except Exception as e:

    print(
        f"❌ Erro Azure Blob: {e}"
    )

    raise

✅ Cliente Azure Blob inicializado.


In [5]:
# =========================
# API IBGE
# =========================

url = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/estados"
)

response = requests.get(
    url,
    timeout=60
)

response.raise_for_status()

data = response.json()

print(
    f"✅ Estados encontrados: {len(data)}"
)

✅ Estados encontrados: 27


In [6]:
# =========================
# DataFrame
# =========================

df = pd.json_normalize(data)

df = df.rename(
    columns={
        "id": "estado_id",
        "sigla": "estado_sigla",
        "nome": "estado_nome",
        "regiao.id": "regiao_id",
        "regiao.sigla": "regiao_sigla",
        "regiao.nome": "regiao_nome"
    }
)

df["_ingested_at"] = (
    dt.datetime.now(
        dt.timezone.utc
    ).isoformat()
)

df.head()

,estado_id,estado_sigla,estado_nome,regiao_id,regiao_sigla,regiao_nome,_ingested_at
0,11,RO,Rondônia,1,N,Norte,2026-08-23T23:31:56.167802+00:00
1,12,AC,Acre,1,N,Norte,2026-08-23T23:31:56.167802+00:00
2,13,AM,Amazonas,1,N,Norte,2026-08-23T23:31:56.167802+00:00
3,14,RR,Roraima,1,N,Norte,2026-08-23T23:31:56.167802+00:00
4,15,PA,Pará,1,N,Norte,2026-08-23T23:31:56.167802+00:00


In [7]:
# =========================
# Salvar parquet
# =========================

temp_dir = Path.cwd() / "tmp"

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

date_suffix = dt.datetime.now().strftime(
    "%Y-%m-%d"
)

parquet_file = (
    temp_dir
    / f"{date_suffix}_ibge_estados.parquet"
)

df.to_parquet(
    parquet_file,
    index=False
)

print(
    f"✅ Parquet criado: {parquet_file}"
)

✅ Parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-23_ibge_estados.parquet


In [9]:
# =========================
# Upload Bronze
# =========================

blob_name = (
    #f"ibge/estados/"
    f"{date_suffix}_ibge_estados.parquet"
)

blob_client = (
    blob_service_client.get_blob_client(
        container=BRONZE_CONTAINER,
        blob=blob_name
    )
)

with open(
    parquet_file,
    "rb"
) as data:

    blob_client.upload_blob(
        data,
        overwrite=True
    )

print(
    f"✅ Upload concluído: "
    f"{blob_name}"
)

✅ Upload concluído: 2026-08-23_ibge_estados.parquet
